# 04 — Traffic Forecasting Data Preparation

## Objective
Prepare the Luzern UTD19 traffic measurements for short-term
traffic-flow forecasting at 15-minute and 30-minute horizons.

The preparation pipeline includes:
1. Data quality inspection
2. Detector selection
3. Timestamp reconstruction
4. Regular 3-minute time series
5. Feature engineering
6. Forecast target generation
7. Chronological train/validation/test split
8. Processed dataset export

In [1]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np

In [2]:
DATA_PATH = Path.home() / "Datasets" / "luzern_raw.parquet"

assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"

con = duckdb.connect()

print(f"Dataset: {DATA_PATH}")

Dataset: /home/bagii/Datasets/luzern_raw.parquet


In [3]:
schema = con.execute("""
    DESCRIBE SELECT *
    FROM read_parquet(?)
""", [str(DATA_PATH)]).df()

display(schema)

,column_name,column_type,null,key,default,extra
0,day,DATE,YES,None,None,None
1,interval,BIGINT,YES,None,None,None
2,detid,VARCHAR,YES,None,None,None
3,flow,BIGINT,YES,None,None,None
4,occ,DOUBLE,YES,None,None,None
5,error,BIGINT,YES,None,None,None
6,city,VARCHAR,YES,None,None,None
7,speed,VARCHAR,YES,None,None,None


In [4]:
quality = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT detid) AS detectors,
        MIN(day) AS first_day,
        MAX(day) AS last_day,

        COUNT(*) FILTER (WHERE flow IS NULL) AS missing_flow,
        COUNT(*) FILTER (WHERE occ IS NULL) AS missing_occ,

        COUNT(*) FILTER (WHERE error IS NULL) AS unflagged_rows,
        COUNT(*) FILTER (WHERE error IS NOT NULL) AS flagged_rows,

        COUNT(*) FILTER (WHERE interval = 86400) AS boundary_intervals,

        MIN(flow) AS min_flow,
        MAX(flow) AS max_flow,
        MIN(occ) AS min_occ,
        MAX(occ) AS max_occ
    FROM read_parquet(?)
""", [str(DATA_PATH)]).df()

display(quality.T)

,0
total_rows,26878297
detectors,158
first_day,2015-01-01 00:00:00
last_day,2015-12-31 00:00:00
missing_flow,0
missing_occ,0
unflagged_rows,17812356
flagged_rows,9065941
boundary_intervals,56026
min_flow,0


In [5]:
duplicates = con.execute("""
    SELECT
        detid,
        day,
        interval,
        COUNT(*) AS n
    FROM read_parquet(?)
    GROUP BY detid, day, interval
    HAVING COUNT(*) > 1
    ORDER BY n DESC, detid, day, interval
""", [str(DATA_PATH)]).df()

print("Duplicate groups:", len(duplicates))
display(duplicates.head(20))

Duplicate groups: 16


,detid,day,interval,n
0,ig11FD205_D2,2015-07-10,86400,2
1,ig11FD205_D81,2015-07-10,86400,2
2,ig11FD205_D87,2015-07-10,86400,2
3,ig11FD205_D88,2015-07-10,86400,2
4,ig11FD205_D89,2015-07-10,86400,2
5,ig11FD205_D90,2015-07-10,86400,2
6,ig11FD205_D91,2015-07-10,86400,2
7,ig11FD205_D92,2015-07-10,86400,2
8,ig11FD208_D1,2015-07-07,63720,2
9,ig11FD208_D1,2015-07-09,28800,2


In [6]:
suspicious = con.execute("""
    SELECT
        error,
        COUNT(*) AS total,

        COUNT(*) FILTER (
            WHERE flow = 99999
        ) AS flow_99999,

        COUNT(*) FILTER (
            WHERE occ > 1
        ) AS invalid_occ,

        COUNT(*) FILTER (
            WHERE interval = 86400
        ) AS boundary_intervals

    FROM read_parquet(?)
    GROUP BY error
    ORDER BY error NULLS FIRST
""", [str(DATA_PATH)]).df()

display(suspicious)

,error,total,flow_99999,invalid_occ,boundary_intervals
0,<NA>,17812356,0,0,31215
1,1,9065941,117805,76020,24811


In [7]:
duplicate_details = con.execute("""
    SELECT
        day,
        interval,
        detid,
        flow,
        occ,
        error
    FROM read_parquet(?)
    WHERE (detid, day, interval) IN (
        SELECT detid, day, interval
        FROM read_parquet(?)
        GROUP BY detid, day, interval
        HAVING COUNT(*) > 1
    )
    ORDER BY detid, day, interval
""", [str(DATA_PATH), str(DATA_PATH)]).df()

display(duplicate_details)

,day,interval,detid,flow,occ,error
0,2015-07-10,86400,ig11FD205_D2,280,0.035000,1
1,2015-07-10,86400,ig11FD205_D2,480,0.070000,1
2,2015-07-10,86400,ig11FD205_D81,20,0.000000,1
3,2015-07-10,86400,ig11FD205_D81,0,0.000000,1
4,2015-07-10,86400,ig11FD205_D87,80,0.000000,1
5,2015-07-10,86400,ig11FD205_D87,160,0.035000,1
6,2015-07-10,86400,ig11FD205_D88,80,0.000000,1
7,2015-07-10,86400,ig11FD205_D88,60,0.010000,1
8,2015-07-10,86400,ig11FD205_D89,400,0.050000,1
9,2015-07-10,86400,ig11FD205_D89,240,0.045000,1


In [8]:
TRAIN_END = pd.Timestamp("2015-09-13")
VAL_END = pd.Timestamp("2015-11-07")

print("Train: 2015-01-01 to", TRAIN_END.date())
print("Validation:", TRAIN_END.date(), "to", VAL_END.date())
print("Test:", VAL_END.date(), "to 2016-01-01")

Train: 2015-01-01 to 2015-09-13
Validation: 2015-09-13 to 2015-11-07
Test: 2015-11-07 to 2016-01-01


In [9]:
detector_quality = con.execute("""
    WITH daily_quality AS (
        SELECT
            detid,
            day,
            COUNT(DISTINCT interval) AS valid_intervals
        FROM read_parquet(?)
        WHERE day < ?
          AND error IS NULL
          AND flow IS NOT NULL
          AND occ BETWEEN 0 AND 1
          AND flow BETWEEN 0 AND 99998
          AND interval BETWEEN 0 AND 86400
          AND interval % 180 = 0
        GROUP BY detid, day
    )
    SELECT
        detid,
        COUNT(*) FILTER (
            WHERE valid_intervals >= 400
        ) AS reliable_days,
        SUM(valid_intervals) AS valid_observations
    FROM daily_quality
    GROUP BY detid
    HAVING COUNT(*) FILTER (
        WHERE valid_intervals >= 400
    ) >= 175
    ORDER BY reliable_days DESC, valid_observations DESC
""", [str(DATA_PATH), TRAIN_END.date()]).df()

selected_detectors = detector_quality["detid"].tolist()

print("Selected detectors:", len(selected_detectors))
display(detector_quality.head(10))

Selected detectors: 79


,detid,reliable_days,valid_observations
0,ig11FD103_D3,251,115691.0
1,ig11FD109_D28,251,114427.0
2,ig11FD116_D35,251,113462.0
3,ig11FD202_D33,251,112752.0
4,ig11FD106_D10,251,112698.0
5,ig11FD202_D2,251,111893.0
6,ig11FD104_D5,251,111208.0
7,ig11FD202_D35,251,110867.0
8,ig11FD116_D25,251,110532.0
9,ig11FD203_D11,251,110224.0


In [10]:
con.register(
    "selected_detector_ids",
    pd.DataFrame({"detid": selected_detectors})
)

timestamp_collisions = con.execute("""
    WITH reconstructed AS (
        SELECT
            r.detid,
            CAST(r.day AS TIMESTAMP)
                + r.interval * INTERVAL '1 second' AS timestamp,
            r.day,
            r.interval,
            r.flow,
            r.occ,
            r.error
        FROM read_parquet(?) AS r
        INNER JOIN selected_detector_ids AS s
            ON r.detid = s.detid
        WHERE r.interval BETWEEN 0 AND 86400
    )
    SELECT
        detid,
        timestamp,
        COUNT(*) AS records,
        COUNT(*) FILTER (
            WHERE error IS NULL
        ) AS unflagged_records
    FROM reconstructed
    GROUP BY detid, timestamp
    HAVING COUNT(*) > 1
    ORDER BY timestamp, detid
""", [str(DATA_PATH)]).df()

print("Timestamp collision groups:", len(timestamp_collisions))
display(timestamp_collisions.head(20))

Timestamp collision groups: 8


,detid,timestamp,records,unflagged_records
0,ig11FD208_D1,2015-07-07 17:42:00,2,0
1,ig11FD208_D2,2015-07-07 17:42:00,2,0
2,ig11FD208_D3,2015-07-07 17:42:00,2,0
3,ig11FD208_D4,2015-07-07 17:42:00,2,0
4,ig11FD208_D1,2015-07-09 08:00:00,2,0
5,ig11FD208_D2,2015-07-09 08:00:00,2,0
6,ig11FD208_D3,2015-07-09 08:00:00,2,0
7,ig11FD208_D4,2015-07-09 08:00:00,2,0


In [11]:
if timestamp_collisions.empty:
    print("No timestamp collisions detected.")
else:
    collision_summary = (
        timestamp_collisions
        .assign(
            quality_class=lambda df: np.select(
                [
                    df["unflagged_records"] == 0,
                    df["unflagged_records"] == 1,
                    df["unflagged_records"] > 1,
                ],
                [
                    "All flagged",
                    "Exactly one unflagged",
                    "Multiple unflagged",
                ],
                default="Unknown"
            )
        )
        .groupby("quality_class")
        .agg(
            collision_groups=("detid", "size"),
            total_records=("records", "sum")
        )
    )

    display(collision_summary)

,collision_groups,total_records
quality_class,,
All flagged,8,16


In [12]:
# Load only measurements from selected detectors
df = pd.read_parquet(
    DATA_PATH,
    columns=["day", "interval", "detid", "flow", "occ", "error"],
    filters=[("detid", "in", selected_detectors)]
)

# Reconstruct timestamps
df["timestamp"] = (
    pd.to_datetime(df["day"])
    + pd.to_timedelta(df["interval"], unit="s")
)

# Keep only correctly aligned 3-minute intervals
df = df[
    df["interval"].between(0, 86400)
    & (df["interval"] % 180 == 0)
].copy()

# Mark valid measurements
df["is_valid"] = (
    df["error"].isna()
    & df["flow"].between(0, 99998)
    & df["occ"].between(0, 1)
)

# Resolve duplicate detector/timestamp records
# Prefer valid measurements if duplicates exist
df = (
    df.sort_values("is_valid", ascending=False)
      .drop_duplicates(subset=["detid", "timestamp"], keep="first")
      .sort_values(["detid", "timestamp"])
      .reset_index(drop=True)
)

# Invalid measurements become missing values
df.loc[~df["is_valid"], ["flow", "occ"]] = np.nan

df = df[
    ["detid", "timestamp", "flow", "occ", "is_valid"]
]

print("Cleaned measurements:", len(df))
display(df.head())

Cleaned measurements: 13818902


,detid,timestamp,flow,occ,is_valid
0,ig11FD100_D3,2015-01-01 00:03:00,180.0,0.015,True
1,ig11FD100_D3,2015-01-01 00:06:00,100.0,0.010,True
2,ig11FD100_D3,2015-01-01 00:09:00,160.0,0.015,True
3,ig11FD100_D3,2015-01-01 00:12:00,NaN,NaN,False
4,ig11FD100_D3,2015-01-01 00:15:00,NaN,NaN,False


In [13]:
# Complete timeline for 2015
timestamps = pd.date_range(
    start="2015-01-01",
    end="2015-12-31 23:57:00",
    freq="3min"
)

# Every detector must have every timestamp
full_index = pd.MultiIndex.from_product(
    [selected_detectors, timestamps],
    names=["detid", "timestamp"]
)

# Align observations to the complete timeline
df_regular = (
    df.set_index(["detid", "timestamp"])
      .reindex(full_index)
      .reset_index()
)

# Distinguish absent rows from flagged measurements
df_regular["is_observed"] = df_regular["is_valid"].notna()
df_regular["is_valid"] = (
    df_regular["is_valid"]
    .fillna(False)
    .astype(bool)
)

# Free memory
del df

print("Regular time series:", df_regular.shape)
display(df_regular.head())

Regular time series: (13840800, 6)


,detid,timestamp,flow,occ,is_valid,is_observed
0,ig11FD103_D3,2015-01-01 00:00:00,NaN,NaN,False,False
1,ig11FD103_D3,2015-01-01 00:03:00,300.0,0.075,True,True
2,ig11FD103_D3,2015-01-01 00:06:00,180.0,0.045,True,True
3,ig11FD103_D3,2015-01-01 00:09:00,280.0,0.085,True,True
4,ig11FD103_D3,2015-01-01 00:12:00,320.0,0.100,True,True


In [14]:
grid_quality = pd.DataFrame({
    "total_slots": [len(df_regular)],
    "detectors": [df_regular["detid"].nunique()],
    "observed_slots": [df_regular["is_observed"].sum()],
    "valid_slots": [df_regular["is_valid"].sum()],
    "missing_slots": [(~df_regular["is_observed"]).sum()],
    "flagged_or_invalid_slots": [
        (df_regular["is_observed"] & ~df_regular["is_valid"]).sum()
    ]
})

display(grid_quality.T)

expected_slots = len(selected_detectors) * 365 * 480

assert len(df_regular) == expected_slots
assert not df_regular.duplicated(["detid", "timestamp"]).any()

# Verify timestamp spacing for every detector
assert (
    df_regular.groupby("detid")["timestamp"]
    .diff()
    .dropna()
    .eq(pd.Timedelta(minutes=3))
    .all()
)

print("Expected slots:", expected_slots)
print("Grid validation passed!")

,0
total_slots,13840800
detectors,79
observed_slots,13818823
valid_slots,12109240
missing_slots,21977
flagged_or_invalid_slots,1709583


Expected slots: 13840800
Grid validation passed!


In [15]:
df_features = df_regular.copy()

# Each detector is an independent time series
df_features = df_features.sort_values(
    ["detid", "timestamp"]
).reset_index(drop=True)

grouped = df_features.groupby("detid", sort=False)

# Calendar features
df_features["hour"] = df_features["timestamp"].dt.hour
df_features["weekday"] = df_features["timestamp"].dt.dayofweek

# Historical traffic flow
for lag in [1, 5, 10]:
    df_features[f"flow_lag_{lag}"] = grouped["flow"].shift(lag)

# Historical occupancy
for lag in [1, 5]:
    df_features[f"occ_lag_{lag}"] = grouped["occ"].shift(lag)

# Rolling averages, including the current observation
for window in [5, 10]:
    df_features[f"flow_roll_mean_{window}"] = (
        grouped["flow"]
        .transform(lambda x: x.rolling(window, min_periods=1).mean())
    )

    # Count observations contributing to each average
    df_features[f"flow_roll_count_{window}"] = (
        grouped["flow"]
        .transform(lambda x: x.rolling(window, min_periods=1).count())
    )

# Future traffic-flow targets
df_features["target_flow_15m"] = grouped["flow"].shift(-5)
df_features["target_flow_30m"] = grouped["flow"].shift(-10)

# Chronological splits
df_features["split"] = np.select(
    [
        df_features["timestamp"] < TRAIN_END,
        df_features["timestamp"] < VAL_END
    ],
    ["train", "validation"],
    default="test"
)

print("Forecasting dataset:", df_features.shape)

display(df_features.head(15))

Forecasting dataset: (13840800, 20)


,detid,timestamp,flow,occ,is_valid,is_observed,hour,weekday,flow_lag_1,flow_lag_5,flow_lag_10,occ_lag_1,occ_lag_5,flow_roll_mean_5,flow_roll_count_5,flow_roll_mean_10,flow_roll_count_10,target_flow_15m,target_flow_30m,split
0,ig11FD100_D3,2015-01-01 00:00:00,NaN,NaN,False,False,0,3,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,220.0,train
1,ig11FD100_D3,2015-01-01 00:03:00,180.0,0.015,True,True,0,3,NaN,NaN,NaN,NaN,NaN,180.000000,1.0,180.000000,1.0,100.0,280.0,train
2,ig11FD100_D3,2015-01-01 00:06:00,100.0,0.010,True,True,0,3,180.0,NaN,NaN,0.015,NaN,140.000000,2.0,140.000000,2.0,120.0,140.0,train
3,ig11FD100_D3,2015-01-01 00:09:00,160.0,0.015,True,True,0,3,100.0,NaN,NaN,0.010,NaN,146.666667,3.0,146.666667,3.0,180.0,280.0,train
4,ig11FD100_D3,2015-01-01 00:12:00,NaN,NaN,False,True,0,3,160.0,NaN,NaN,0.015,NaN,146.666667,3.0,146.666667,3.0,180.0,220.0,train
5,ig11FD100_D3,2015-01-01 00:15:00,NaN,NaN,False,True,0,3,NaN,NaN,NaN,NaN,NaN,146.666667,3.0,146.666667,3.0,220.0,200.0,train
6,ig11FD100_D3,2015-01-01 00:18:00,100.0,0.010,True,True,0,3,NaN,180.0,NaN,NaN,0.015,120.000000,3.0,135.000000,4.0,280.0,180.0,train
7,ig11FD100_D3,2015-01-01 00:21:00,120.0,0.015,True,True,0,3,100.0,100.0,NaN,0.010,0.010,126.666667,3.0,132.000000,5.0,140.0,280.0,train
8,ig11FD100_D3,2015-01-01 00:24:00,180.0,0.015,True,True,0,3,120.0,160.0,NaN,0.015,0.015,133.333333,3.0,140.000000,6.0,280.0,140.0,train
9,ig11FD100_D3,2015-01-01 00:27:00,180.0,0.020,True,True,0,3,180.0,NaN,NaN,0.015,NaN,145.000000,4.0,145.714286,7.0,220.0,180.0,train


In [16]:
# Check target values against timestamps shifted by the horizon
for steps, horizon in [(5, 15), (10, 30)]:
    grouped = df_features.groupby("detid", sort=False)

    future_timestamps = grouped["timestamp"].shift(-steps)

    aligned = (
        future_timestamps.isna()
        | (
            future_timestamps - df_features["timestamp"]
            == pd.Timedelta(minutes=horizon)
        )
    )

    assert aligned.all(), f"{horizon}-minute alignment failed"

    print(f"{horizon}-minute target alignment passed!")

15-minute target alignment passed!
30-minute target alignment passed!


In [17]:
required_features = [
    "flow",
    "occ",
    "flow_lag_1",
    "flow_lag_5",
    "flow_lag_10",
    "occ_lag_1",
    "occ_lag_5",
    "flow_roll_mean_5",
    "flow_roll_mean_10",
]

# Verify all required inputs exist
valid_inputs = (
    df_features["is_valid"]
    & df_features[required_features].notna().all(axis=1)
    & (df_features["flow_roll_count_5"] == 5)
    & (df_features["flow_roll_count_10"] == 10)
)

# A target must belong to the same chronological split
for horizon, steps in [(15, 5), (30, 10)]:
    future_split = (
        df_features.groupby("detid", sort=False)["split"]
        .shift(-steps)
    )

    df_features[f"usable_{horizon}m"] = (
        valid_inputs
        & df_features[f"target_flow_{horizon}m"].notna()
        & df_features["split"].eq(future_split)
    )

print("Usability flags created!")

Usability flags created!


In [18]:
forecast_quality = (
    df_features.groupby("split")
    .agg(
        total_rows=("detid", "size"),
        valid_current=("is_valid", "sum"),
        usable_15m=("usable_15m", "sum"),
        usable_30m=("usable_30m", "sum"),
    )
    .reindex(["train", "validation", "test"])
)

display(forecast_quality)

print("\nUsable example percentages:")
display(
    (
        forecast_quality[["usable_15m", "usable_30m"]]
        .div(forecast_quality["total_rows"], axis=0)
        * 100
    ).round(2)
)

,total_rows,valid_current,usable_15m,usable_30m
split,,,,
train,9669600,8448465,6261815,6248652
validation,2085600,1807869,1343994,1340719
test,2085600,1852906,1363284,1360422



Usable example percentages:


,usable_15m,usable_30m
split,,
train,64.76,64.62
validation,64.44,64.28
test,65.37,65.23


In [19]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "luzern_forecasting_v1.parquet"

df_features.to_parquet(
    output_path,
    engine="pyarrow",
    index=False,
    compression="zstd"
)

print(f"Saved dataset to: {output_path}")
print(f"Rows: {len(df_features):,}")
print(f"Detectors: {df_features['detid'].nunique()}")

Saved dataset to: ../data/processed/luzern_forecasting_v1.parquet
Rows: 13,840,800
Detectors: 79
